# Setup

In [ ]:
from dotenv import load_dotenv
import pandas as pd
import sys
import json
sys.path.append("../../../src")

from processor.model.interface.model_factory import get_llm

load_dotenv("../../.env")

model_path = "o3"
o3 = get_llm("o3")("o3")

In [ ]:
DATA_SOURCE = "environment"
with open(f"../../../benchmark/sources (kramabench)/{DATA_SOURCE}.json") as f:
    benchmark = json.load(f)
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
def get_table_repr(table: pd.DataFrame, table_id: str):
    table_repr = f"Table {table_id}:\ncol: {" | ".join(list(table.columns))}"
    if len(table) > 0:
        row_idx = 1
        for _, data in table.iterrows():
            str_data = [str(i) for i in data]
            table_repr += (
                f"\n- row {row_idx}: {" | ".join(str_data)}"
            )
            row_idx += 1
    return table_repr

def get_table_repr_driver(tables: dict[str, pd.DataFrame]):
    final_repr = ""
    for table_id, table in tables.items():
        final_repr += get_table_repr(table, table_id)
        final_repr += "\n"
    return final_repr.strip()

def get_qa_prompt(question: str, relevant_tables: dict[str, pd.DataFrame]):
    return f"""Given this question: ```{question}``` 

Please answer the question based on the following relevant tables:
{get_table_repr_driver(relevant_tables)}

End your answer with this format: ANSWER: ..."""

# Evaluation

In [ ]:
from processor.model.llm_message import LLMMessage

answers = read_jsonl("answers-environment.jsonl")
IDX = 19
print(benchmark[IDX]['query'])
print(benchmark[IDX]['answer'])
print(answers[IDX]['answer'])

In [ ]:
from tqdm import tqdm
import time

from processor.model.llm_message import LLMMessage

answers: list[dict] = []
for i in [IDX]:
    curr_benchmark = benchmark[i]
    print(f"Handling question {i}: {curr_benchmark['query']}")
    start = time.time()
    curr_benchmark = benchmark[i]
    relevant_tables = [i for i in curr_benchmark['data_sources'] if i.endswith(".csv")]
    relevant_tables_final: dict[str, pd.DataFrame] = dict()
    for relevant_table_path in relevant_tables:
        relevant_tables_final[relevant_table_path] = pd.read_csv(
            f"../../../data_src/{DATA_SOURCE}/dataset/{relevant_table_path}"
        )
    prompt = get_qa_prompt(
        curr_benchmark['query'],
        relevant_tables_final
    )
    o3_messages = [
        LLMMessage(
            role='user',
            content=prompt
        )
    ]
    try:
        answer = o3.chat(o3_messages)
    except Exception as e:
        answer = str(e)
    print(f"=> Answer: {answer}")
    answers.append({
        "question": curr_benchmark['query'],
        "answer": answer,
        "ground_truth": o3_messages,
    })
    write_jsonl("answers.jsonl", answers)
    end = time.time()
    print(f"Total time for question {i}: {end-start} seconds.")